In [23]:
from dotenv import load_dotenv

load_dotenv()

from openai import OpenAI, AsyncOpenAI
from llm_utils import llm_call, llm_call_async
from typing import List
import asyncio
from lib.tools import parse_md_json

async_client = AsyncOpenAI()
sync_client = OpenAI()

# 웹 검색 기능을 포함한 LLM 호출 함수 선언
async def llm_search_async(prompt: str, model: str = "gpt-4.1") -> str:
    response = await async_client.responses.create(
        model=model,
        input=prompt,
        tools=[{"type": "web_search"}],
    )
    return response.output_text

async def run_llm_parallel(prompt_details):
    tasks = [
        llm_search_async(item['user_prompt'], item['model'])
        for item in prompt_details
    ]
    responses = await asyncio.gather(*tasks)
    return responses

In [24]:
test_res = llm_call("너 뭐야", model="gpt-4o")
print(f' ==> [Line 1]: \033[38;2;216;234;16m[test_res]\033[0m({type(test_res).__name__}) = \033[38;2;19;48;199m{test_res}\033[0m')


 ==> [Line 1]: [test_res](str) = 저는 AI 기반 언어 모델입니다. 정보 제공, 질문 답변, 대화 등의 기능을 수행할 수 있습니다. 어떻게 도와드릴까요?


# 자소서 질문 생성기

## Phase 1 · JD 분석

### 1-1 워커 실행 함수 및 모델 라우팅 준비

In [25]:
import json
from lib.phase1 import run_jd_workers
from llm_utils.utils import llm_call_async

# 1. 모델명에 따라 일반 LLM 호출과 검색(Search) LLM 호출을 라우팅하는 래퍼 함수
async def notebook_llm_async_fn(prompt: str, model: str) -> str:
    print(f"[{model}] 호출 중...")
    if model == "gpt-4.1": 
        # 첫 번째 셀에 정의된 llm_search_async 사용
        return await llm_search_async(prompt, model=model)
    else:
        # 워커 기본 모델은 llm_call_async 사용
        return await llm_call_async(prompt, model=model)


### 1-2 JD 샘플 텍스트 로드

In [26]:
# 2. JD 샘플 텍스트 로드
with open('JD-sample.md', 'r', encoding='utf-8') as f:
    jd_text = f.read()

### 1-3 Phase 1 병렬 실행 및 결과 확인

In [27]:
# 3. Phase 1 파이프라인 병렬 실행
phase1_results = await run_jd_workers(jd_text, notebook_llm_async_fn)

[gpt-4o] 호출 중...
[gpt-4o] 호출 중...
[gpt-4o] 호출 중...
[gpt-4.1] 호출 중...
[gpt-4.1] 호출 중...
gpt-4o 완료
gpt-4o 완료
gpt-4o 완료


In [28]:
# 4. 결과 확인
print("\n" + "="*50)
print("✅ Phase 1 분석 결과 (JSON Wrapper)")
print("="*50)
for res in phase1_results:
    print(f"\n▶ Agent ID: {res['agent_id'].upper()}")
    print(json.dumps(res, ensure_ascii=False, indent=2))


✅ Phase 1 분석 결과 (JSON Wrapper)

▶ Agent ID: ROLE
{
  "company_name": "삼성전자 DS부문 AI센터",
  "job_title": "SW개발",
  "agent_id": "role",
  "payload": {
    "roles": [
      {
        "role_name": "반도체 도메인 특화 AI 모델, Agent 시스템/서비스, Platform 연구 및 개발",
        "required_skills": [
          "AI 모델링",
          "Agent 시스템 개발",
          "Platform 연구"
        ],
        "question_type": "직무적합"
      },
      {
        "role_name": "반도체 공정/수율/설비 데이터 기반 AI Agent 아키텍처 연구 및 설계",
        "required_skills": [
          "데이터 분석",
          "AI 아키텍처 설계"
        ],
        "question_type": "의사결정"
      },
      {
        "role_name": "LLM 기반 추론 구조 및 멀티 Agent 협업 시스템 개발",
        "required_skills": [
          "LLM 이해",
          "멀티 Agent 시스템 개발"
        ],
        "question_type": "기술깊이"
      },
      {
        "role_name": "반도체 분석 Workflow 자동화 Agent 설계 및 고도화",
        "required_skills": [
          "Workflow 자동화",
          "Agent 설계"
        ],
        "question_type": "기술깊이"
      },
      {
        

## Phase 2 · 자소서 분석 - 프롬프트 체이닝

### 2-1 자기소개서 각 항목 분류

In [29]:
import importlib
import lib.phase2 as phase2
from lib.tools import parse_md_json

importlib.reload(phase2)
from lib.phase2 import classify_personal_statement_sections

file_path = './essay-sample.md'
with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()

print("=" * 50)
print("=== [STEP 1] 자기소개서 각 항목 분류 ===")
print("=" * 50)
personal_statement_res = classify_personal_statement_sections(content)
personal_statement_list = parse_md_json(personal_statement_res)
if isinstance(personal_statement_list, dict):
    personal_statement_list = [personal_statement_list]

print(f' ==> [Line 11]: \033[38;2;85;130;182m[personal_statement_list]\033[0m({type(personal_statement_list).__name__}) = \n\033[38;2;248;199;101m{personal_statement_list}\033[0m')

=== [STEP 1] 자기소개서 각 항목 분류 ===
 ==> [Line 11]: [personal_statement_list](list) = 
[{'question': '삼성전자를 지원한 이유와 입사 후 회사에서 이루고 싶은 꿈을 기술하십시오.', 'answer': '[40시간의 업무 병목을 해소한 설계 역량, 지연 없는 AI 서빙으로 이어가다] 삼성전자 DS부문의 Autonomous Fab이 현장에 안착하려면 데이터를 지연 없이 처리하는 서빙 플랫폼의 성능이 보장되어야 합니다. 수많은 설비에서 발생하는 데이터를 실시간으로 분석하고 결과를 반환하는 과정에서 응답 지연이 발생하면 제조 공정 전체의 병목으로 이어집니다. 저는 삼성전자 네트워크사업부 연계 프로젝트에서 레거시 엑셀 중심으로 진행되던 업무를 웹 기반 협업 시스템으로 재설계해 40시간의 작업 시간을 1시간으로 단축한 경험이 있습니다. 현장 실무자의 업무 병목을 해결한 역량을 바탕으로 DS부문의 AI 플랫폼 안정성을 책임지기 위해 지원했습니다. 입사 후 목표는 현장 엔지니어들이 지연 없이 사용할 수 있는 AI Agent 백엔드 환경을 구축하는 것입니다. 이를 위해 두 단계의 기술적 로드맵을 실행하겠습니다. 첫째, 반도체 공정에서 발생하는 방대한 비정형 데이터의 특성을 분석하고, 이에 최적화된 데이터베이스 I/O 통제 및 인덱스 설계 기준을 수립하겠습니다. 둘째, 무거운 AI 연산 결과가 대규모 트래픽 속에서도 안정적으로 반환될 수 있도록 캐싱 전략과 아키텍처를 단계적으로 고도화하겠습니다. 지속적인 트래픽 모니터링과 쿼리 튜닝을 통해 데이터 병목을 사전에 차단하고 견고한 실시간 서빙 인프라를 완성에 기여 하겠습니다.'}, {'question': '본인의 성장과정을 간략히 기술하되 현재의 자신에게 가장 큰 영향을 끼친 사건, 인물 등을 포함하여 기술하시기 바랍니다.', 'answer': "[통제 가능한 루틴으로 불확실성을 극복하다] 저의 성장을 이끈 핵심 동력은 명확한 규칙을 세우고 이를 매일 지켜내는 '

### 2-2 각 JD와 자기소개서 항목을 비교해서 매칭 & 분석

In [30]:
from pprint import pprint
from lib.phase2 import analyze_and_match_essay

sample_jd_list = [item for item in phase1_results if item["agent_id"] == "role"]
if not sample_jd_list:
    raise ValueError("phase1_results에서 agent_id='role' 결과를 찾지 못했습니다.")

target_jd = sample_jd_list[0]

print("=" * 50)
print("=== [STEP 2] analyze_and_match_essay ===")
print("=" * 50)
analyzed_result = analyze_and_match_essay(
    js_list=target_jd,
    personal_statement_list=personal_statement_list,
    client=sync_client,
    model="gpt-4o-mini",
)
pprint(analyzed_result)

=== [STEP 2] analyze_and_match_essay ===
{'items': [{'answer_text': '[40시간의 업무 병목을 해소한 설계 역량, 지연 없는 AI 서빙으로 이어가다] 삼성전자 '
                           'DS부문의 Autonomous Fab이 현장에 안착하려면 데이터를 지연 없이 처리하는 서빙 '
                           '플랫폼의 성능이 보장되어야 합니다. 수많은 설비에서 발생하는 데이터를 실시간으로 분석하고 '
                           '결과를 반환하는 과정에서 응답 지연이 발생하면 제조 공정 전체의 병목으로 이어집니다. 저는 '
                           '삼성전자 네트워크사업부 연계 프로젝트에서 레거시 엑셀 중심으로 진행되던 업무를 웹 기반 '
                           '협업 시스템으로 재설계해 40시간의 작업 시간을 1시간으로 단축한 경험이 있습니다. 현장 '
                           '실무자의 업무 병목을 해결한 역량을 바탕으로 DS부문의 AI 플랫폼 안정성을 책임지기 위해 '
                           '지원했습니다. 입사 후 목표는 현장 엔지니어들이 지연 없이 사용할 수 있는 AI Agent '
                           '백엔드 환경을 구축하는 것입니다. 이를 위해 두 단계의 기술적 로드맵을 실행하겠습니다. '
                           '첫째, 반도체 공정에서 발생하는 방대한 비정형 데이터의 특성을 분석하고, 이에 최적화된 '
                           '데이터베이스 I/O 통제 및 인덱스 설계 기준을 수립하겠습니다. 둘째, 무거운 AI 연산 '
                           '결과가 대규모 트래픽 속에서도 안정적으로 반환될 수 있도록 캐싱 전략과 아키텍처를 '
  

### 2-3 문항별 분석 결과를 바탕으로 면접 질문 생성에 바로 쓸 주제, 검증 포인트, 리스크 포인트, 꼬리질문 주제를 정리하는 단계

In [31]:
from lib.phase2 import build_question_context

print("\n" + "=" * 50)
print("=== [STEP 3] build_question_context ===")
print("=" * 50)
context_result = build_question_context(
    analyzed_result=analyzed_result,
    client=sync_client,
    model="gpt-4o-mini",
)
print(context_result)


=== [STEP 3] build_question_context ===
{'target': {'company': '삼성전자', 'division': 'DS부문 AI센터', 'role': 'SW개발'}, 'items': [{'question_id': 'Q1', 'question_text': '삼성전자를 지원한 이유와 입사 후 회사에서 이루고 싶은 꿈을 기술하십시오.', 'answer_text': '[40시간의 업무 병목을 해소한 설계 역량, 지연 없는 AI 서빙으로 이어가다] 삼성전자 DS부문의 Autonomous Fab이 현장에 안착하려면 데이터를 지연 없이 처리하는 서빙 플랫폼의 성능이 보장되어야 합니다. 수많은 설비에서 발생하는 데이터를 실시간으로 분석하고 결과를 반환하는 과정에서 응답 지연이 발생하면 제조 공정 전체의 병목으로 이어집니다. 저는 삼성전자 네트워크사업부 연계 프로젝트에서 레거시 엑셀 중심으로 진행되던 업무를 웹 기반 협업 시스템으로 재설계해 40시간의 작업 시간을 1시간으로 단축한 경험이 있습니다. 현장 실무자의 업무 병목을 해결한 역량을 바탕으로 DS부문의 AI 플랫폼 안정성을 책임지기 위해 지원했습니다. 입사 후 목표는 현장 엔지니어들이 지연 없이 사용할 수 있는 AI Agent 백엔드 환경을 구축하는 것입니다. 이를 위해 두 단계의 기술적 로드맵을 실행하겠습니다. 첫째, 반도체 공정에서 발생하는 방대한 비정형 데이터의 특성을 분석하고, 이에 최적화된 데이터베이스 I/O 통제 및 인덱스 설계 기준을 수립하겠습니다. 둘째, 무거운 AI 연산 결과가 대규모 트래픽 속에서도 안정적으로 반환될 수 있도록 캐싱 전략과 아키텍처를 단계적으로 고도화하겠습니다. 지속적인 트래픽 모니터링과 쿼리 튜닝을 통해 데이터 병목을 사전에 차단하고 견고한 실시간 서빙 인프라를 완성에 기여 하겠습니다.', 'item_type': 'motivation_vision', 'question_intent': ['지원동기', '입사 후 목표'], 'matched_jd': [

### 2-4 모든 문항 결과를 한데 모아 전체 JD 커버리지, 공통 약점, 우선 질문 주제를 계산해 Phase2 최종 JSON으로 만드는 단계.

In [32]:
from lib.phase2 import aggregate_phase2_results

print("\n" + "=" * 50)
print("=== [STEP 4] aggregate_phase2_results ===")
print("=" * 50)
final_output = aggregate_phase2_results(
    js_list=target_jd,
    contextualized_result=context_result,
    client=sync_client,
    model="gpt-4o-mini",
)

phase2_result = final_output
pprint(final_output)


=== [STEP 4] aggregate_phase2_results ===
{'global_context': {'global_risks': ['기술적 로드맵의 현실성 부족',
                                     '이해관계자와의 협업 필요성 간과',
                                     '경험의 과장 가능성',
                                     '경험이 다소 추상적으로 느껴질 수 있음',
                                     '구체적인 기술 능력에 대한 명시적 예제가 부족함',
                                     '변화에 대한 적응력을 과장할 가능성 있음',
                                     '기술적 설명의 깊이가 부족할 수 있음',
                                     '주요 기술에 대한 직접적 경험 부족 우려',
                                     '이론적 주장에 대한 실증적 사례 부족',
                                     '특정 도메인(반도체)에 대한 경험 부족 가능성',
                                     'AI Agent 시스템에 대한 깊이 있는 지식 부족 우려',
                                     '경험의 구체적 성과 부족'],
                    'missing_jd_ids': ['R2',
                                       'R4',
                                       'R7',
                                       'R9',
                                       'R1

## Phase 3 · 질문 산출

In [33]:
from lib.phase3 import phase3
phase3()

Phase 3


## Phase 4 · 평가-최적화 루프

In [ ]:
import os
import sys
from llm_utils.utils import llm_call

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

EVALUATOR_PROMPT = """
당신은 {job_category} 직무 면접관(실무자, 팀장, HR)입니다.

다음 면접 질문을 평가해줘.

[질문]
{question}

평가 기준:
1. 꼬리질문으로 이어질 수 있는가
2. 질문이 구체적인가
3. JD에서 요구하는 역량과 연결되는가
4. 자기소개서 기반으로 답할 수 있는가
5. 실제 면접관이 할 법한 질문인가

판정 기준:
- 모든 평가 기준이 충족되면 PASS
- 하나라도 부족하면 FAIL

응답 양식:
평가 결과: PASS 또는 FAIL
문제점 및 개선 방향:
- PASS면 "없음"
- FAIL이면 부족한 점과 어떻게 고치면 좋을지 구체적으로 작성
"""

def phase4(question: str, job_category: str) -> str:
    prompt = EVALUATOR_PROMPT.format(
        job_category=job_category,
        question=question,
    )
    return llm_call(prompt)


# questions = [
    'AI 데이터 파이프라인 구축 경험에서 본인이 직접 설계하거나 구현한 핵심 컴포넌트와 그 기여도를 수치나 명확한 근거로 상세히 설명해보세요.',
    '실시간 서빙 아키텍처를 최적화할 때 선택한 기술 스택(Caching, Message Queue, Stream Processing 등)의 도입 결정 기준과 각각이 성능 병목을 어떻게 해소하는지 내부 메커니즘까지 구체적으로 말해보세요.',
    '파이프라인 운영 도중 예기치 못한 데이터 스키마 변동이나 대용량 트래픽 처리 실패와 같은 위기 상황을 실제로 겪었을 때, 트레이드오프를 어떻게 판단하고 시스템 신뢰성을 지키기 위해 어떤 조처를 했습니까?',
    '최신 AI Agent 서비스 플랫폼 트렌드를 고려할 때, 삼성전자 DS부문의 제조/반도체 특화 환경에 실시간 서빙 파이프라인을 어떤 방식으로 적용·확장하여 경쟁사 대비 차별화할 수 있다고 생각합니까?',
# ]

def get_pass_questions(questions: list[str], job_category: str) -> list[str]:
    pass_questions = []

    for question in questions:
        result = phase4(question, job_category)
        print(f"[평가 중] {question}")
        print(result)
        print("-" * 80)

        if "평가 결과: PASS" in result:
            pass_questions.append(question)

    return pass_questions


# pass_questions = get_pass_questions(questions, "백엔드개발자")

# print("\nPASS 질문만 추린 결과:")
# for i, question in enumerate(pass_questions, 1):
#     print(f"{i}. {question}")

Phase 4


## Phase 5 · 최종 출력

In [ ]:
from lib.phase5 import phase5
phase5()

Phase 5
